In [ ]:
import os
import sys
import time

import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import scanpy as sc
sc.settings.savefig_args = {'dpi': 300}

import matplotlib
import matplotlib.font_manager as fm
fm.fontManager.addfont('../../Arial.ttf')
matplotlib.rcParams['font.size'] = 16.0
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Arial']
%matplotlib inline

%load_ext autoreload
%autoreload 2

# Eykthyr imports
sys.path.insert(0, '.')
from eykthyr.eykthyr import Eykthyr, load_anndata
import eykthyr.plotting as epl

In [ ]:
# Load the pre-computed Eykthyr object
# Adjust the path to where the Eykthyr object was saved via e.save_anndata()
ey = load_anndata('../eykthyrtest3.h5ad')

# Convenience aliases matching the original notebook
datasets = ey.perturbed_X   # perturbed metagene embeddings + gene info
samples = ey.datasetnames
display(samples)
datasets

In [ ]:
datasets = [sc.read_h5ad(
    f'../processed_data/misar_popari/trained_K16_i200_differential_best_slices_simulation_{s}_X_norms.h5ad'
                        ) for s in samples]
ey.perturbed_X = datasets

# Subfig A

In [ ]:
leiden_labels = {'0': 'Midbrain 1',
                 '1': 'Cartilage 1',
                 '2': 'Cartilage 2',
                 '3': 'Pallium',
                 '4': 'Mesenchyme 1',
                 '5': 'Hindbrain',
                 '6': 'Muscle',
                 '7': 'Mesenchyme 2',
                 '8': 'Mesenchyme 3',
                 '9': 'Midbrain 2'
                }

datasets[0].obs['labeled_leiden'] = [leiden_labels[c] for c in datasets[0].obs['original_leiden']]

leiden_labels = {'0': 'Mesenchyme 1',
                 '1': 'Cartilage 1',
                 '2': 'Hindbrain 1',
                 '3': 'Cartilage 2',
                 '4': 'Subpallium',
                 '5': 'Hindbrain 2',
                 '6': 'DPallm',
                 '7': 'Muscle',
                 '8': 'Mesenchyme 2',
                 '9': 'Pallium'
                }
datasets[1].obs['labeled_leiden'] = [leiden_labels[c] for c in datasets[1].obs['original_leiden']]

leiden_labels = {'0': 'Midbrain',
                 '1': 'Diencephalon',
                 '2': 'Cartilage',
                 '3': 'Midbrain',
                 '4': 'Mesenchyme',
                 '5': 'DPallm',
                 '6': 'DPallv',
                 '7': 'Subpallium',
                 '8': 'Mesenchyme',
                 '9': 'Muscle'
                }
datasets[2].obs['labeled_leiden'] = [leiden_labels[c] for c in datasets[2].obs['original_leiden']]

leiden_labels = {'0': 'DPallm 1',
                 '1': 'Hindbrain 2',
                 '2': 'DPallm 2',
                 '3': 'DPallv',
                 '4': 'Diencephalon 2',
                 '5': 'Subpallium',
                 '6': 'Hindbrain 1',
                 '7': 'Cartilage 1',
                 '8': 'Diencephalon 1',
                 '9': 'Cartilage 2'
                }
datasets[3].obs['labeled_leiden'] = [leiden_labels[c] for c in datasets[3].obs['original_leiden']]

In [ ]:
leiden_colors = {'Cartilage': '#b2df8a',
                 'Cartilage 1': '#e41a1c',
                 'Cartilage 2': '#a21a1c',
                 'Mesenchyme': '#ff7f00',
                 'Mesenchyme 1': '#ff7f00',
                 'Mesenchyme 2': '#ff7f00',
                 'Mesenchyme 3': '#b87ed3',
                 'Hindbrain': '#ffff33',
                 'Hindbrain 1': '#ffff33',
                 'Hindbrain 2': '#aaaa13',
                 'Midbrain': '#a6cee3',
                 'Midbrain 1': '#a6cee3',
                 'Midbrain 2': '#a6cee3',
                 'Muscle': '#33a02c',
                 'Pallium': '#377eb8',
                 'DPallm': '#fb9a99',
                 'DPallm 1': '#ff7f00',
                 'DPallm 2': '#ffaf40',
                 'DPallv': '#e31a1c',
                 'Subpallium': '#fdbf6f',
                 'Diencephalon': '#1f78b4',
                 'Diencephalon 1': '#377eb8',
                 'Diencephalon 2': '#074e88'}

for d in datasets:
    d.uns['labeled_leiden_colors'] = [leiden_colors[a] for a in sorted(d.obs['labeled_leiden'].unique().astype('str'))]

In [ ]:
matplotlib.rcParams['font.size'] = 14.0
sc.pl.spatial(datasets[2], color='labeled_leiden', spot_size=1, frameon=False, title='', legend_fontsize='small', 
              save='panel_2A.svg'
             )

# Subfig B

In [ ]:
origrdf = pd.read_csv('../misar_pallium_metrics.csv')
origrdf = origrdf.set_index('Embedding')

In [ ]:
cordf = pd.read_csv('../co_misar_benchmarks.csv')
cordf = cordf.set_index('Embedding')
cordf = cordf.drop(index=['Metric Type'])

In [ ]:
coTFs = [tf.split('_')[1] for tf in cordf.index]
sharedTFs = [tf for tf in origrdf.index if tf.split('_')[2] in coTFs]
rdf = origrdf.loc[sharedTFs]
rdf = rdf.sort_values(by='Batch correction')
X = rdf['Batch correction'][-20:].index
X = [x.split('_')[2] for x in X]
height = rdf['Batch correction'][-20:].values

In [ ]:
matplotlib.rcParams['font.size'] = 16.0
fig, ax = plt.subplots(1,1,figsize=(5,2))
markerline, stemlines, baseline = ax.stem(X,height,orientation='vertical', bottom=0.5, 
                                          linefmt='#1c9099', 
                                          markerfmt='C0o', basefmt='w-')
ax.set_ylabel('score')
ax.tick_params(axis='x', labelrotation=60)
markerline.set_markerfacecolor('#1c9099')
markerline.set_markeredgecolor('#1c9099')
markerline.set_markersize(8)
plt.savefig(f'Panel_2B.svg', bbox_inches='tight')

# Subfigs D & E

In [ ]:
# TF activity is stored in ey.TF
# ey.TF[i] corresponds to datasets[i] (same order as ey.datasetnames)
# tfadatas2 in original notebook = ey.TF

# Subset to common TF var_names across samples (matching original notebook behavior)
tfadatas2 = ey.TF

# Gene expression is stored in ey.RNA (already normalized and log-transformed)
# exadatas in original notebook corresponded to E13_5-S1 (index 1) and E15_5-S2 (index 2)
# Map: exadatas[1] -> ey.RNA[2] (E15_5-S2)
exadatas = ey.RNA

In [ ]:
datasets[2].obs['X_11'] = datasets[2].obsm['X'][:,11]
sc.pl.spatial(datasets[2], color='X_11', spot_size=1, frameon=False, title='Metagene 11 expression', cmap='viridis', colorbar_loc=None,
             save='Panel_2E_left.svg')

In [ ]:
tf = 'Msx1'
sc.pl.spatial(tfadatas2[2],color=tf,spot_size=1, cmap='viridis', title=f'{tf} TF openness', frameon=False, colorbar_loc=None, 
              save=f'Panel_2D_right.svg')
sc.pl.spatial(exadatas[2],color=tf,spot_size=1, cmap='viridis', title=f'{tf} gene expression', frameon=False, colorbar_loc=None, 
             save=f'Panel_2D_left.svg')

datasets[2].obs[f'{tf}_X_11'] = datasets[2].obsm['X'][:,11] - datasets[2].obsm[f'X_{tf}_dropout'][:,11] 
sc.pl.spatial(datasets[2], color=f'{tf}_X_11', spot_size=1, cmap='bwr', vmin=-0.04, vmax=0.04, frameon=False, 
              title=f'{tf} M11 delta', colorbar_loc=None, save=f'Panel_2E_right.svg')

# Subfig I & J

In [ ]:
# Compute PAGA and force-directed graph layout for all datasets
# Uses normalized_X embedding (from Eykthyr/Popari metagenes)
# sc.pp.neighbors(ey.perturbed_X[2], use_rep='normalized_X')
# epl.prep_paga(ey, 'original_leiden')
sc.tl.paga(ey.perturbed_X[2], groups='original_leiden')
sc.pl.paga(ey.perturbed_X[2])
sc.tl.draw_graph(ey.perturbed_X[2], init_pos='umap', random_state=123)
# Note: embedding stored in obsm['X_draw_graph_fr'] (force-directed, FR layout)

In [ ]:
sc.pl.draw_graph(ey.perturbed_X[2], color='original_leiden', legend_loc="on data")

## Spatial simulation arrow plots

In [ ]:
# In the Eykthyr framework, datasets already contain the obsm['X'] metagene embedding.
# datasets_X from the original notebook is equivalent to datasets here.
# Color info transfer for simulation plots:
for d in datasets:
    d.uns['labeled_leiden_colors'] = [leiden_colors[a] for a in sorted(d.obs['labeled_leiden'].unique().astype('str'))]
    d.uns['original_leiden_colors'] = d.uns['labeled_leiden_colors']

In [ ]:
# Subset to pallium region (DPallv=6, Subpallium=7) with spatial coordinates filter
# This matches the pallium_only subset from the original notebook
pallium_mask = ((datasets[2].obs['original_leiden'] == '6') |
                (datasets[2].obs['original_leiden'] == '7'))
spatial_mask = ((datasets[2].obsm['spatial'][:,0] > 7) &
                (datasets[2].obsm['spatial'][:,0] < 35) &
                (datasets[2].obsm['spatial'][:,1] < 27))
pallium_only_ad = datasets[2][pallium_mask & spatial_mask].copy()
pallium_only_ad.uns['original_leiden_colors'] = pallium_only_ad.uns['labeled_leiden_colors']

# Create a temporary Eykthyr object for the pallium subset
ey_pallium = Eykthyr()
ey_pallium.perturbed_X = [pallium_only_ad]
ey_pallium.num_metagenes = ey.num_metagenes

# Flip spatial y-axis for arrow visualization (matching original notebook)
ey_pallium.perturbed_X[0].obsm['spatial_2'] = ey_pallium.perturbed_X[0].obsm['spatial'].copy()
ey_pallium.perturbed_X[0].obsm['spatial_2'][:,1] = ey_pallium.perturbed_X[0].obsm['spatial_2'][:,1] * -1
pallium_only_ad

In [ ]:
TFs = ['Msx1']

# Direct dropout simulation using Eykthyr plotting API
# Equivalent to the Oracle-based simulation in the original notebook (Cell 41)
epl.umap_spatial_simulation(
    ey_pallium,
    TFs[0],
    n_grid=14,
    min_masses=[1, 0.01],
    scales=[20, 0.5],
    embeddings=['spatial_2', 'X_draw_graph_fa'],
    n_neighbors=[50, 50],
    cluster_name='original_leiden',
    show_plots=[True, True],
)

In [ ]:
from scipy.stats import zscore
def normalize(obsm):
    normalized_embeddings = zscore(obsm)
    nan_mask = np.isnan(normalized_embeddings)
    normalized_embeddings[nan_mask] = 0
    return normalized_embeddings

In [ ]:
multiplier = 15
for tf in ['Msx1']:
    datasets[2].obsm[f'{tf}_KO_mult'] = datasets[2].obsm['X'] + multiplier * (datasets[2].obsm[f'X_{tf}_dropout'] - datasets[2].obsm['X'])
    datasets[2].obsm[f'normalized_{tf}_KO_mult'] = normalize(datasets[2].obsm[f'{tf}_KO_mult'])
    sc.pp.neighbors(datasets[2], use_rep=f'normalized_{tf}_KO_mult', key_added='normTFKOmult')
    fig, axs = plt.subplots(1,3,figsize=(20,10))
    for res, ax in zip([0.65, 0.7, 0.75], axs):
        sc.tl.leiden(datasets[2], neighbors_key='normTFKOmult', key_added=f'mult_leiden_{tf}_{res}', resolution=res)
        sc.pl.spatial(datasets[2], color=f'mult_leiden_{tf}_{res}', spot_size=1, ax=ax, show=False, title=f'mult_leiden_{tf}_{res}')
    plt.show()

In [ ]:
# Amplified perturbation simulation (multiplied KO, equivalent to original Cell 42)
# Temporarily set the KO_mult embedding as the dropout embedding for visualization
multiplier_sim = 15
for tf in ['Msx1']:
    # Use normalized KO_mult embedding for transition probability estimation
    # Store temporarily under the dropout key so plotting API can use it
    pallium_mask2 = ((datasets[2].obs['original_leiden'] == '6') |
                     (datasets[2].obs['original_leiden'] == '7'))
    spatial_mask2 = ((datasets[2].obsm['spatial'][:,0] > 7) &
                     (datasets[2].obsm['spatial'][:,0] < 35) &
                     (datasets[2].obsm['spatial'][:,1] < 27))
    pallium_msx_ad = datasets[2][pallium_mask2 & spatial_mask2].copy()
    pallium_msx_ad.uns['original_leiden_colors'] = pallium_msx_ad.uns['labeled_leiden_colors']
    pallium_msx_ad.obsm['spatial_2'] = pallium_msx_ad.obsm['spatial'].copy()
    pallium_msx_ad.obsm['spatial_2'][:,1] = pallium_msx_ad.obsm['spatial_2'][:,1] * -1

    # Replace dropout embedding with the amplified version
    orig_dropout = pallium_msx_ad.obsm[f'X_{tf}_dropout'].copy()
    orig_norm_dropout = pallium_msx_ad.obsm[f'normalized_X_{tf}_dropout'].copy()
    pallium_msx_ad.obsm[f'X_{tf}_dropout'] = pallium_msx_ad.obsm[f'{tf}_KO_mult']
    pallium_msx_ad.obsm[f'normalized_X_{tf}_dropout'] = normalize(pallium_msx_ad.obsm[f'{tf}_KO_mult'])

    ey_msx_pallium = Eykthyr()
    ey_msx_pallium.perturbed_X = [pallium_msx_ad]
    ey_msx_pallium.num_metagenes = ey.num_metagenes

    epl.umap_spatial_simulation(
        ey_msx_pallium,
        tf,
        n_grid=14,
        min_masses=[1, 0.01],
        scales=[30, 30],
        embeddings=['spatial_2', 'X_draw_graph_fr'],
        n_neighbors=[50, 50],
        cluster_name='original_leiden',
        show_plots=[True, True],
    )

    # Additional spatial leiden change visualization
    fig3, ax3 = plt.subplots(1,2,figsize=[8.5, 7])
    sc.pl.spatial(pallium_msx_ad, color='labeled_leiden', spot_size=1, ax=ax3[0], show=False, frameon=False, legend_loc=None, title='')
    sc.pl.spatial(pallium_msx_ad, color=f'mult_leiden_{tf}_0.75', spot_size=1, ax=ax3[1], show=False, frameon=False, legend_loc=None, title='')
    plt.savefig(f'Panels_2IJ.svg')
    plt.show()

# Supplementary Fig

In [ ]:
import gseapy as gp
from gseapy import barplot, dotplot

In [ ]:
for i in range(ey.num_metagenes):
    datasets[2].obs[f'X_{i}'] = datasets[2].obsm['X'][:,i]
    sc.pl.spatial(datasets[2], color=f'X_{i}', spot_size=1, frameon=False, title=f'Metagene {i} expression', cmap='Reds',
                  save=f'misarmetagene{i}.png')

In [ ]:
ldf = pd.DataFrame(index=[str(s) for s in datasets[2].obs['labeled_leiden'].unique()], columns=[f'm{j}' for j in range(ey.num_metagenes)], dtype=float)
for l in datasets[2].obs['labeled_leiden'].unique():
    ldf.loc[l,:] = datasets[2][datasets[2].obs['labeled_leiden'] == l].obsm['X'].mean(axis=0)
ldf = ldf.div(ldf.sum(axis=0), axis=1)

In [ ]:
import seaborn as sns
fig, ax = plt.subplots(1,1,figsize=(8,4))
ax.set_title('Mean metagene expression')
sns.heatmap(ldf, vmax=0.4, cmap='Reds', ax=ax)
fig.savefig('figures/misarmetageneheatmap.png', dpi=300, bbox_inches='tight')

# Subfig F

In [ ]:
import gseapy as gp
from gseapy import barplot, dotplot

def get_response_dfs(tf, datasets, datasetnames):
    """
    Compute per-cell-type gene-level response to TF perturbation.
    
    Uses the metagene perturbation (X_{tf}_dropout vs X) and the metagene-to-gene
    matrix (stored in datasets[i].uns['M'][datasetname]) to project into gene space.
    
    Parameters
    ----------
    tf : str
        TF name to compute response for.
    datasets : list of AnnData
        Perturbed datasets (ey.perturbed_X), each with obsm['X'] and obsm['X_{tf}_dropout'].
    datasetnames : list of str
        Dataset names corresponding to each entry in datasets.
    
    Returns
    -------
    list of pd.DataFrame
        One DataFrame per dataset; rows=genes, columns=cell types.
        Positive values = genes upregulated upon TF dropout in that cell type.
    """
    response_dfs = [pd.DataFrame(index=d.var.index, columns=d.obs['labeled_leiden'].unique())
                    for d in datasets]
    for d, df, dname in zip(datasets, response_dfs, datasetnames):
        d.obsm[f'{tf}_mse'] = d.obsm['X'] - d.obsm[f'X_{tf}_dropout']
        K = d.obsm['X'].shape[1]
        ct_df = pd.DataFrame(index=[f'X_{i}' for i in range(K)],
                             columns=[ct for ct in d.obs['labeled_leiden'].unique()])
        for ct in d.obs['labeled_leiden'].unique():
            ct_mse = d[d.obs['labeled_leiden'] == ct].obsm[f'{tf}_mse'].mean(axis=0)
            ct_df.loc[:, ct] = ct_mse
            # Project metagene changes to gene space using per-dataset M matrix
            gene_ct_mse = np.matmul(d.uns['M'][dname], ct_mse)
            df.loc[:, ct] = gene_ct_mse
        absmax = max(abs(ct_df.to_numpy().min()), abs(ct_df.to_numpy().max()))
        fig, ax = plt.subplots(1,1,figsize=(12,8))
        ax.set_yticks(np.arange(len(ct_df.columns)), labels=ct_df.columns)
        ax.set_xticks(np.arange(len(ct_df.index)), labels=[f'{k}' for k in range(len(ct_df.index))])
        ax.imshow(ct_df.to_numpy().astype(float).T, cmap='bwr', vmin=-absmax, vmax=absmax)
        print(ct_df.to_numpy().min(), ct_df.to_numpy().max())
        plt.savefig(f'{tf}_rdf.svg')
        plt.show()
    return response_dfs

ans = get_response_dfs('Msx1', [datasets[2]], [samples[2]])

In [ ]:
# NOTE: This was created in the notebooks: misarseq_celloracle.ipynb and CellOracle_GRN_part3-Copy1.ipynb
msxgenes = []
for line in open('../msx_genes_brain2.csv','r').readlines():
    msxgenes.append(line.strip())

In [ ]:
possiblegenes = []
for line in open('../possible_genes.csv','r').readlines():
    possiblegenes.append(line.strip())

In [ ]:
possiblesortedind = np.asarray(ans[0][ans[0].index.isin(possiblegenes)].sort_values(
    by='DPallv', ascending=True).index).astype(str)
l2 = []
for g in msxgenes:
    lil = np.where(possiblesortedind == g)
    if len(lil[0]) > 0:
        l2.append(lil[0][0])

In [ ]:
possiblemsxgenes = [g for g in msxgenes if g in possiblesortedind]

In [ ]:
ans[0]['abs_DPallv'] = [abs(v) for v in ans[0]['DPallv']]
ans[0]['abs_Subpallium'] = [abs(v) for v in ans[0]['Subpallium']]
ans[0]['abs_DPallm'] = [abs(v) for v in ans[0]['DPallm']]
ans[0]['abs_all'] = ans[0]['abs_DPallv'] + ans[0]['abs_DPallm'] + ans[0]['abs_Subpallium']

In [ ]:
from scipy.stats import ttest_1samp
col = ['abs_all']
possiblesortedind = np.asarray(ans[0].sort_values(
    by=col, ascending=False).index).astype(str)
l2 = []
for g in msxgenes:
    lil = np.where(possiblesortedind == g)
    if len(lil[0]) > 0:
        l2.append(lil[0][0])
from scipy.stats import ttest_1samp
print(ttest_1samp(l2, len(possiblesortedind) / 2).pvalue)
bestrankss2 = [l2]
bestcols = ['ranking']
fig, ax = plt.subplots(1,1,figsize=(3,5))
bp = ax.boxplot(bestrankss2, patch_artist=True, labels=bestcols)
plt.setp(bp['boxes'], color='#1c9099')
plt.setp(bp['medians'], color='red')
plt.setp(bp['whiskers'], color='black')
plt.setp(bp['fliers'], color='black')
for patch in bp['boxes']:
    patch.set_edgecolor('black')
ax.tick_params(axis='x', labelrotation=60)
ax.axhline(len(possiblesortedind) / 2, linestyle='--', color='black')
plt.savefig(f'Panel_2F.svg', bbox_inches='tight')

# Subfig K

In [ ]:
def do_gsea(glist, cell_type, background, get_dotplot=True):
    enr = gp.enrichr(gene_list=glist,
                 gene_sets=['GO_Biological_Process_2023','Reactome_2022'],
                 organism='mouse',
                 outdir=None,
                )
    enr.results.sort_values(by='Adjusted P-value')
    if get_dotplot == True:
        ax = dotplot(enr.results,
              column="Adjusted P-value",
              x='Gene_set',
              size=10,
              top_term=5,
              figsize=(3,5),
              title=f"GSEA {cell_type}",
              xticklabels_rot=45,
              show_ring=True,
              marker='o',
             )
    ax2 = barplot(enr.results,
              column="Adjusted P-value",
              group='Gene_set',
              size=10,
              top_term=5,
              figsize=(3,5),
              color=['red','green','blue'],
              title=f'GSEA {cell_type}',
             )
    return enr

In [ ]:
ans = get_response_dfs('Msx1', [datasets[2]], [samples[2]])
topk = []
k = 200
bigk = k * len(ans[0].columns)
for column in ans[0].columns:
    topkrows = ans[0].sort_values(by=column, ascending=True).index[:bigk]
    for row in topkrows:
        topk.append((ans[0].loc[row,column], column, row))
topk.sort()
topkchanged = topk[:bigk]
glists = [[g[2] for g in topkchanged if g[1] == ct] for ct in ans[0].columns]
print([len(gl) for gl in glists])
enrs = []
for i, ct in enumerate(ans[0].columns):
    if len(glists[i]) > k:
        time.sleep(100)
        enrs.append(do_gsea(glists[i][:k], ct, datasets[2].var_names.to_list(), get_dotplot=False))

In [ ]:
dpallmenr = enrs[0]
GOenr = dpallmenr.results[dpallmenr.results['Gene_set'] == 'GO_Biological_Process_2023'].sort_values(by='Adjusted P-value')
reactomeenr = dpallmenr.results[dpallmenr.results['Gene_set'] == 'Reactome_2022'].sort_values(by='Adjusted P-value')
X1 = GOenr['Term'].values[:5].tolist()
X2 = reactomeenr['Term'].values[:5].tolist()
X1 = [' '.join(x.split()[:-1]) for x in X1]
X2 = [' '.join(x.split()[:-1]) for x in X2]
X1[2] = f'{X1[2]} (GO)'
X1[4] = f'{X1[4]} (GO)'
X2[1] = f'{X2[1]} (Reactome)'
X2[2] = f'{X2[2]} (Reactome)'
X1.reverse()
X2.reverse()

In [ ]:
height1 = GOenr['Adjusted P-value'].values[:5].tolist()
height2 = reactomeenr['Adjusted P-value'].values[:5].tolist()
height1 = [np.log10(1 / float(h)) for h in height1]
height2 = [np.log10(1 / float(h)) for h in height2]
height1.reverse()
height2.reverse()

In [ ]:
marker_sizes1 = GOenr['Odds Ratio'].values[:5].tolist()
marker_sizes2 = reactomeenr['Odds Ratio'].values[:5].tolist()
marker_sizes1.reverse()
marker_sizes2.reverse()
desired_max = 20
desired_min = 5
min_marker_size = min(min(marker_sizes1), min(marker_sizes2))
max_marker_size = max(max(marker_sizes1), max(marker_sizes2))
print(f'min Odds Ratio: {min_marker_size}\nmax Odds Ratio: {max_marker_size}')
marker_sizes1 = [(m - min_marker_size) + desired_min for m in marker_sizes1]
marker_sizes2 = [(m - min_marker_size) + desired_min for m in marker_sizes2]
max_marker_size = max(max(marker_sizes1), max(marker_sizes2))
marker_sizes1 = [(m / max_marker_size) * (desired_max - desired_min) for m in marker_sizes1]
marker_sizes2 = [(m / max_marker_size) * (desired_max - desired_min) for m in marker_sizes2]
marker_sizes1 = [m + desired_min for m in marker_sizes1]
marker_sizes2 = [m + desired_min for m in marker_sizes2]
print(marker_sizes1)

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(15,6))
for ms, x, h in zip(marker_sizes1, X1, height1):
    markerline, stemlines, baseline = ax.stem(x, h, orientation='horizontal', markerfmt='C0o', basefmt='w-',
                                          label='GO', linefmt='#1b9e77')
    markerline.set_markerfacecolor('#1b9e77')
    markerline.set_markeredgecolor('#1b9e77')
    markerline.set_markersize(ms)
for ms, x, h in zip(marker_sizes2, X2, height2):
    markerline2, stemlines2, baseline2 = ax.stem(x, h, orientation='horizontal', markerfmt='C0o', basefmt='w-',
                                             label='Reactome', linefmt='#7570b3')
    markerline2.set_markerfacecolor('#7570b3')
    markerline2.set_markeredgecolor('#7570b3')
    markerline2.set_markersize(ms)
ax.set_xlabel(r'$-\log$(adj. p-val.)')
ax.figure.savefig(f'Panel_2K.svg')

# Subfig H is in eykthyr-ma-compbio/Nrg_isoform_spatial.ipynb

# Subfig C is in eykthyr-ma-compbio/Ablation_plots.ipynb

# Subfig G was created using Genome Browser